# Frozen-flow fix test notebook

This notebook tests the fixed solver and fixed kernels without touching your original files.

In [2]:
import json
import numpy as np
import pandas as pd

from frg_flow_fixed import FRGFlowSolver, BareVertexFromInteraction
from frg_kernel_fixed import FlowConfig, compute_pp_kernel, compute_ph_kernel, compute_phc_kernel


## 1. Paste the same setup you used in your existing benchmark

Expected objects after this cell:
- `patchsets`
- `interaction`
- optional `diagnoser` (or set `diagnoser = None`)

In [21]:
import numpy as np
import matplotlib.pyplot as plt

from noninteracting import KagomeKaneMeleSOC, KagomeStaggerFlux, KagomeNagaosa
from patching import FSPatcher
from interaction import BareExtendedHubbard
from channels import ChannelDecomposer
from frg_kernel_fixed import (
    FlowConfig,
    compute_pp_kernel,
    compute_ph_kernel,
    compute_phc_kernel,
)
from kagome_order_diagnosis import KagomeOrderDiagnoser

# Optional: only if your current Module 4 solver exists and works
try:
    from frg_flow_fixed import FRGFlowSolver, BareVertexFromInteraction
    HAS_FLOW_SOLVER = True
except Exception:
    HAS_FLOW_SOLVER = False
    print("frg_flow.py not available or not importable; flow tests will be notebook-design only.")


In [23]:
def build_spin_patchsets(model, mu, band_index=1, Npatch=12, grid_size=220):
    patch_up = FSPatcher(
        model,
        band_index=band_index,
        mu=mu,
        Npatch=Npatch,
        grid_size=grid_size,
        orbital_slice=slice(0, 3),
        gauge_fix="parallel_transport",
    ).build()

    patch_dn = FSPatcher(
        model,
        band_index=band_index,
        mu=mu,
        Npatch=Npatch,
        grid_size=grid_size,
        orbital_slice=slice(3, 6),
        gauge_fix="parallel_transport",
    ).build()

    return {"up": patch_up, "dn": patch_dn}


def build_reference_model_and_patchsets(
    *,
    model_name="kane_mele",
    parameters=None,
    filling=0.50,
    Npatch=12,
    grid_size=220,
    band_index=1,
):
    if model_name == "kane_mele":
        if parameters is None:
            parameters = {"t": 1.0, "l1": 0.0, "l2": 0.0}
        model = KagomeKaneMeleSOC(parameters=parameters, spin=True, B=0.0)
    elif model_name == "stagger_flux":
        if parameters is None:
            parameters = {"t": 1.0, "phi": 0.0}
        model = KagomeStaggerFlux(parameters=parameters, spin=True, B=0.0)
    elif model_name == "nagaosa":
        if parameters is None:
            parameters = {"t": 1.0, "phi": 0.0}
        model = KagomeNagaosa(parameters=parameters, spin=True, B=0.0)
    else:
        raise ValueError("Unknown model_name")

    mu = model.EF_from_filling(filling)
    patchsets = build_spin_patchsets(model, mu, band_index=band_index, Npatch=Npatch, grid_size=grid_size)
    return model, mu, patchsets


def make_interaction(model, U, V):
    return BareExtendedHubbard.from_kagome_model(model, U=U, V=V)


def block_tensor_norm(T):
    arr = np.asarray(T)
    return float(np.max(np.abs(arr)))


def kernel_norm(kernel):
    return float(np.max(np.abs(kernel.matrix)))


def relative_diff(a, b, floor=1e-14):
    a = np.asarray(a)
    b = np.asarray(b)
    denom = max(float(np.max(np.abs(a))), float(np.max(np.abs(b))), floor)
    return float(np.max(np.abs(a - b)) / denom)


def build_gamma_tensors(interaction, patchsets):
    spin_blocks = [
        ("up", "up", "up", "up"),
        ("dn", "dn", "dn", "dn"),
        ("up", "dn", "up", "dn"),
        ("up", "dn", "dn", "up"),
        ("dn", "up", "up", "dn"),
        ("dn", "up", "dn", "up"),
    ]
    out = {}
    for key in spin_blocks:
        s1, s2, s3, s4 = key
        out[key] = interaction.patch_tensor(
            patchsets, s1, s2, s3, s4,
            antisym=True,
            enforce_momentum=False,
        )
    return out


def pick_test_Qs(patchsets, n_pick=6):
    patch_up = patchsets["up"]
    ks = patch_up.patch_k
    idx = np.linspace(0, len(ks)-1, num=min(n_pick, len(ks)), dtype=int)
    return [ks[i] for i in idx]


def compute_basic_channel_set(gamma, patchsets, Q, T=0.20, nfreq=64):
    cfg = FlowConfig(temperature=T, nfreq=nfreq, include_explicit_T_prefactor=True)

    out = {}
    try:
        out["pp_ud_ud"] = compute_pp_kernel(
            gamma, patchsets, Q,
            incoming_spins=("up", "dn"),
            outgoing_spins=("up", "dn"),
            config=cfg,
        )
    except Exception as e:
        out["pp_ud_ud_error"] = e

    try:
        out["ph_uu"] = compute_ph_kernel(
            gamma, patchsets, Q,
            incoming_spins=("up", "up"),
            outgoing_spins=("up", "up"),
            config=cfg,
        )
    except Exception as e:
        out["ph_uu_error"] = e

    try:
        out["ph_dd"] = compute_ph_kernel(
            gamma, patchsets, Q,
            incoming_spins=("dn", "dn"),
            outgoing_spins=("dn", "dn"),
            config=cfg,
        )
    except Exception as e:
        out["ph_dd_error"] = e

    try:
        out["phc_uu"] = compute_phc_kernel(
            gamma, patchsets, Q,
            incoming_spins=("up", "up"),
            outgoing_spins=("up", "up"),
            config=cfg,
        )
    except Exception as e:
        out["phc_uu_error"] = e

    try:
        out["phc_dd"] = compute_phc_kernel(
            gamma, patchsets, Q,
            incoming_spins=("dn", "dn"),
            outgoing_spins=("dn", "dn"),
            config=cfg,
        )
    except Exception as e:
        out["phc_dd_error"] = e

    return out


In [25]:
if HAS_FLOW_SOLVER:
    model, mu, patchsets = build_reference_model_and_patchsets(
        model_name="kane_mele",
        parameters={"t": 1.0, "l1": 0.0, "l2": 0.0},
        filling=0.50,
        Npatch=10,
    )
    interaction = make_interaction(model, U=0.10, V=0.0)
    bare_gamma = BareVertexFromInteraction(interaction, patchsets)
    diagnoser = KagomeOrderDiagnoser(patchsets_by_spin=patchsets)

    solver = FRGFlowSolver(
        patchsets=patchsets,
        bare_gamma=bare_gamma,
        diagnoser=diagnoser,
        T_start=0.40,
        T_stop=0.10,
        n_steps=6,
        nfreq=32,
        diagnose_every=1,
    )
#     history = solver.run()
#     for h in history:
#         print(h.summary_dict())
# else:
#     print("Skip: FRGFlowSolver not importable.")


## 2. Build the fixed solver

In [27]:
solver = FRGFlowSolver(
    patchsets=patchsets,
    bare_gamma=BareVertexFromInteraction(interaction, patchsets),
    diagnoser=diagnoser if 'diagnoser' in globals() else None,
    T_start=0.20,
    T_stop=0.10,
    n_steps=4,
    nfreq=64,
    include_explicit_T_prefactor=True,
    diagnose_every=1,
)

print("spin_blocks =", solver.spin_blocks)
print("n_pp_Q =", len(solver.pp_grid.q_list))
print("n_ph_Q =", len(solver.phd_grid.q_list))
print("n_phc_Q =", len(solver.phc_grid.q_list))


spin_blocks = [('up', 'up', 'up', 'up'), ('dn', 'dn', 'dn', 'dn'), ('up', 'dn', 'up', 'dn'), ('up', 'dn', 'dn', 'up'), ('dn', 'up', 'up', 'dn'), ('dn', 'up', 'dn', 'up')]
n_pp_Q = 53
n_ph_Q = 51
n_phc_Q = 51


## 3. Check that the three channel RHS stores are now nonzero

In [30]:
T0 = float(solver.state.T)
rhs_pp, rhs_phd, rhs_phc = solver.compute_channel_rhs(T0)

def store_stats(store):
    vals = [float(np.max(np.abs(v))) for v in store.values()]
    nz = sum(v > 1e-14 for v in vals)
    return {
        "count": len(vals),
        "nonzero_count": int(nz),
        "max_abs": float(max(vals) if vals else 0.0),
        "min_abs": float(min(vals) if vals else 0.0),
    }

summary = {
    "pp": store_stats(rhs_pp),
    "phd": store_stats(rhs_phd),
    "phc": store_stats(rhs_phc),
}
print(json.dumps(summary, indent=2))


{
  "pp": {
    "count": 318,
    "nonzero_count": 212,
    "max_abs": 0.0635176412991852,
    "min_abs": 0.0
  },
  "phd": {
    "count": 306,
    "nonzero_count": 204,
    "max_abs": 0.060961099596533126,
    "min_abs": 0.0
  },
  "phc": {
    "count": 306,
    "nonzero_count": 204,
    "max_abs": 0.060961099596533126,
    "min_abs": 0.0
  }
}


In [34]:
T0 = float(solver.state.T)
cfg = FlowConfig(temperature=T0, nfreq=16, include_explicit_T_prefactor=True)

Q0 = solver.pp_grid.q_list[0]

ker_pp = compute_pp_kernel(
    solver.gamma_accessor(),
    patchsets,
    Q0,
    incoming_spins=("up", "dn"),
    outgoing_spins=("up", "dn"),
    config=cfg,
    allowed_spin_blocks=solver.spin_blocks,
)

print("pp max abs =", np.max(np.abs(ker_pp.matrix)))
print("pp nonzero count =", np.sum(np.abs(ker_pp.matrix) > 1e-14))

pp max abs = 0.061673803376949256
pp nonzero count = 100


In [36]:
T0 = float(solver.state.T)
cfg = FlowConfig(temperature=T0, nfreq=16, include_explicit_T_prefactor=True)

Q0 = solver.pp_grid.q_list[0]

ker_bad = compute_pp_kernel(
    solver.gamma_accessor(),
    patchsets,
    Q0,
    incoming_spins=("up", "dn"),
    outgoing_spins=("up", "up"),   # 这个会牵涉 forbidden block
    config=cfg,
    allowed_spin_blocks=solver.spin_blocks,
)

print("bad-block pp max abs =", np.max(np.abs(ker_bad.matrix)))
print("bad-block pp nonzero count =", np.sum(np.abs(ker_bad.matrix) > 1e-14))

bad-block pp max abs = 0.0
bad-block pp nonzero count = 0


In [38]:
gamma = solver.gamma_accessor()

test_blocks = [
    (0, "up", 0, "dn", 0, "up", 0, "dn"),  # allowed
    (0, "up", 0, "dn", 0, "up", 0, "up"),  # previously problematic
    (0, "up", 0, "up", 0, "up", 0, "dn"),  # previously problematic
]

for args in test_blocks:
    try:
        val = gamma(*args)
        print(args, "->", val)
    except Exception as e:
        print(args, "-> ERROR:", repr(e))

(0, 'up', 0, 'dn', 0, 'up', 0, 'dn') -> (0.06956668720152745+0j)
(0, 'up', 0, 'dn', 0, 'up', 0, 'up') -> 0j
(0, 'up', 0, 'up', 0, 'up', 0, 'dn') -> 0j


## 4. Check one-step flow is not frozen

In [50]:
before = float(solver.state.channel_norm())
record = solver.step(float(solver.state.T), -1e-4)
after = float(solver.state.channel_norm())

print("before =", before)
print("rhs =", record.rhs_norm)
print("after =", after)
print("substeps =", record.accepted_substeps)
print("max_rel_update =", record.max_rel_update)

RuntimeError: Adaptive step control requested too many substeps. Reduce the temperature spacing or relax max_relative_update.

In [46]:
import importlib
import frg_flow

importlib.reload(frg_flow)

<module 'frg_flow' from 'C:\\Users\\mli826\\Github_rep\\functional-RG-Kagome-\\frg_flow.py'>

In [48]:
import numpy as np
from frg_flow import BareVertexFromInteraction

print("state.channel_norm() =", solver.state.channel_norm())

gamma_bare = BareVertexFromInteraction(interaction, patchsets)

# 采样一些 bare vertex 大小
samples = []
test_spins = [
    ("up","up","up","up"),
    ("dn","dn","dn","dn"),
    ("up","dn","up","dn"),
    ("up","dn","dn","up"),
    ("dn","up","up","dn"),
    ("dn","up","dn","up"),
]

Np = min(4, patchsets["up"].Npatch)   # 只采样前几个patch，避免太慢
for s1,s2,s3,s4 in test_spins:
    for p1 in range(Np):
        for p2 in range(Np):
            for p3 in range(Np):
                p4 = 0
                val = gamma_bare(p1,s1,p2,s2,p3,s3,p4,s4)
                samples.append(abs(val))

print("sampled bare-gamma max =", np.max(samples))
print("sampled bare-gamma mean =", np.mean(samples))
print("sampled bare-gamma min =", np.min(samples))

state.channel_norm() = 0.0
sampled bare-gamma max = 0.06956668720152745
sampled bare-gamma mean = 0.009292461471615575
sampled bare-gamma min = 0.0


In [44]:
T0 = float(solver.state.T)
rhs_pp, rhs_phd, rhs_phc = solver.compute_channel_rhs(T0)

rhs_norm = solver._rhs_norm(rhs_pp, rhs_phd, rhs_phc)
channel_norm = max(solver.state.channel_norm(), 1e-14)

dT = 1e-4
rel_update_current = abs(dT) * rhs_norm / channel_norm

print("rhs_norm =", rhs_norm)
print("channel_norm used in current code =", channel_norm)
print("rel_update_current =", rel_update_current)

rhs_norm = 0.0635176412991852
channel_norm used in current code = 1e-14
rel_update_current = 635176412.991852


## 5. Short run history

In [ ]:
solver2 = FRGFlowSolver(
    patchsets=patchsets,
    bare_gamma=BareVertexFromInteraction(interaction, patchsets),
    diagnoser=diagnoser if 'diagnoser' in globals() else None,
    T_start=0.20,
    T_stop=0.12,
    n_steps=5,
    nfreq=64,
    include_explicit_T_prefactor=True,
    diagnose_every=1,
)

hist = solver2.run()
df = pd.DataFrame([x.summary_dict() for x in hist])
display(df[["step_index", "temperature", "channel_norm", "rhs_norm", "leading_channel_name", "leading_eigenvalue_abs"]])


## 6. Optional sanity check against the original symptom

You should **not** see `rhs_norm = 0` on every step anymore, and `channel_norm` should no longer stay identically zero.